# Exploracao de Dados

Notebook para verificar integridade e estrutura dos dados coletados.

**Fontes disponiveis:**
- **SGS**: Series temporais BCB (Selic, CDI, PTAX, IBC-Br, IGP-M)
- **Expectations**: Expectativas do Relatorio Focus
- **CAGED**: Microdados de emprego formal (MTE)
- **IPEA**: Series agregadas IPEADATA
- **SIDRA**: Series IBGE (IPCA, PIB, etc)
- **Bloomberg**: Dados de mercado (requer terminal)

## 1. Setup

In [ ]:
# Nova API unificada - import unico!
import adb
import adb.core.charting # Registra o accessor .agora

# Visualizacao (opcional, para ajustes finos)
import matplotlib.pyplot as plt

# Query engine para uso direto
qe = adb.QueryEngine()

print("Fontes disponiveis:", adb.available_sources())

## 2. SGS - Series Temporais BCB

In [ ]:
# Indicadores disponiveis
print("Indicadores SGS:")
for ind in adb.sgs.available():
    info = adb.sgs.info(ind)
    print(f"  - {ind}: {info['name']} ({info['frequency']})")

In [ ]:
# Cobertura temporal - indicadores diarios
print("SGS DAILY - Cobertura Temporal")
print("-" * 70)

for ind in adb.sgs.available(frequency='daily'):
    df = adb.sgs.read(ind)
    if not df.empty:
        print(f"{ind:20} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>8,} registros")
    else:
        print(f"{ind:20} | SEM DADOS")

In [ ]:
# Cobertura temporal - indicadores mensais
print("SGS MONTHLY - Cobertura Temporal")
print("-" * 70)

for ind in adb.sgs.available(frequency='monthly'):
    df = adb.sgs.read(ind)
    if not df.empty:
        print(f"{ind:20} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>8,} registros")
    else:
        print(f"{ind:20} | SEM DADOS")

In [ ]:
# Amostra de dados - Selic
df_selic = adb.sgs.read('selic')
print(f"Selic: {len(df_selic):,} registros")
print(f"\nPrimeiros registros:")
display(df_selic.head())
print(f"\nUltimos registros:")
display(df_selic.tail())

In [ ]:
# Visualizacao rapida - Taxas de juros (CDI anualizado)
df_taxas = adb.sgs.read('selic', 'cdi', start='2000')

# CDI anualizado (base 252 dias uteis)
df_taxas['cdi_anual'] = (1 + df_taxas['cdi'] / 100) ** 252 - 1
df_taxas['cdi_anual'] = df_taxas['cdi_anual'] * 100

# Plotagem usando o novo modulo de charting
ax = df_taxas[['selic', 'cdi_anual']].agora.plot(
    title='Selic vs CDI Anualizado (2000+)'
)
ax.set_ylabel('Taxa (% a.a.)')
plt.show()

In [ ]:
# Visualizacao rapida - Cambio
df_cambio = adb.sgs.read('dolar_ptax', 'euro_ptax', start='2015')

ax = df_cambio[['dolar_ptax', 'euro_ptax']].agora.plot(
    title='Taxas de Cambio PTAX (2015+)'
)
ax.set_ylabel('R$')
plt.show()

In [ ]:
# Visualizacao rapida - IBC-Br
df_ibc = adb.sgs.read('ibc_br_bruto', 'ibc_br_dessaz')

ax = df_ibc[['ibc_br_bruto', 'ibc_br_dessaz']].agora.plot(
    title='IBC-Br - Indice de Atividade Economica'
)
ax.set_ylabel('Indice')
plt.show()

## 3. Expectations - Relatorio Focus

In [ ]:
# Indicadores disponiveis
print("Indicadores Expectations:")
for ind in adb.expectations.available():
    info = adb.expectations.info(ind)
    print(f"  - {ind}: {info.get('indicator', 'N/A')} ({info.get('endpoint', 'N/A')})")

In [ ]:
# Cobertura temporal por indicador
print("EXPECTATIONS - Cobertura Temporal")
print("-" * 70)

for ind in adb.expectations.available():
    df = adb.expectations.read(ind)
    if not df.empty:
        print(f"{ind:20} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>8,} registros")
    else:
        print(f"{ind:20} | SEM DADOS")

In [ ]:
# Amostra de dados - IPCA Anual
df_ipca = adb.expectations.read('ipca_anual')
print(f"IPCA Anual: {len(df_ipca):,} registros")
print(f"Colunas: {list(df_ipca.columns)}")
print(f"\nAmostra:")
display(df_ipca.sample(5))

In [ ]:
# Visualizacao rapida - Mediana das expectativas de IPCA
if 'Mediana' in df_ipca.columns:
    mediana_diaria = df_ipca.groupby(df_ipca.index)['Mediana'].mean()
    
    # Converter Series para DataFrame para usar o acessor
    ax = mediana_diaria.to_frame(name='Mediana').agora.plot(
        title='IPCA Anual - Mediana das Expectativas'
    )
    ax.set_xlabel('Data da Pesquisa')
    ax.set_ylabel('Expectativa (%)')
    plt.show()

## 4. IPEA - Series Agregadas

In [ ]:
# Indicadores disponiveis
print("Indicadores IPEA:")
for ind in adb.ipea.available():
    info = adb.ipea.info(ind)
    print(f"  - {ind}: {info['name']}")

In [ ]:
# Cobertura temporal
print("IPEA - Cobertura Temporal")
print("-" * 70)

for ind in adb.ipea.available():
    df = adb.ipea.read(ind)
    if not df.empty:
        print(f"{ind:25} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>6,} registros")
    else:
        print(f"{ind:25} | SEM DADOS")

In [ ]:
# Amostra de dados - Saldo CAGED
df_saldo = adb.ipea.read('caged_saldo')
print(f"Saldo CAGED: {len(df_saldo):,} registros")
print(f"\nUltimos registros:")
display(df_saldo.tail())

In [ ]:
# Visualizacao rapida - Saldo CAGED
ax = df_saldo.agora.plot(
    y='value',
    title='Saldo de Empregos Formais (CAGED)'
)
ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_ylabel('Saldo (pessoas)')

# Formatter manual ainda necessario pois o plotter nao infere automaticamente escala de milhar
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))

plt.show()

## 5. CAGED - Microdados

In [ ]:
# Datasets disponiveis
print("Datasets CAGED:")
for ds, info in adb.caged.info().items():
    print(f"  - {ds}: {info['name']}")

In [ ]:
# Periodos disponiveis
periods = adb.caged.available_periods()
if periods:
    print(f"Periodos disponiveis: {periods[0]} a {periods[-1]}")
    print(f"Total: {len(periods)} meses")
else:
    print("Nenhum dado coletado ainda. Execute: adb.caged.collect()")

In [ ]:
# Amostra de dados - CAGED 2025-10 (se disponivel)
try:
    df_caged = adb.caged.read(year=2025, month=10)
    print(f"CAGED 2025-10: {len(df_caged):,} registros")
    print(f"Colunas: {list(df_caged.columns)}")
    print(f"\nResumo:")
    print(f"  Admissoes: {(df_caged['saldomovimentacao'] == 1).sum():,}")
    print(f"  Desligamentos: {(df_caged['saldomovimentacao'] == -1).sum():,}")
    print(f"  Saldo: {df_caged['saldomovimentacao'].sum():,}")
    print(f"\nAmostra:")
    display(df_caged.sample(5))
except Exception as e:
    print(f"Erro ao carregar dados: {e}")

In [ ]:
# Saldo por UF (usando metodo de agregacao)
try:
    df_saldo_uf = adb.caged.saldo_por_uf(year=2025)
    print("Saldo CAGED por UF - 2025:")
    display(df_saldo_uf.head(10))
except Exception as e:
    print(f"Erro: {e}")

In [ ]:
# Saldo mensal (usando metodo de agregacao)
try:
    df_saldo_mes = adb.caged.saldo_mensal(year=2025)
    print("Saldo CAGED Mensal - 2025:")
    display(df_saldo_mes)
    
    # Visualizacao usando Charting Module
    # O index precisa ser o eixo X (mes) para o grafico de barras funcionar bem automaticamente
    ax = df_saldo_mes.set_index('mes')['saldo'].agora.plot(
        kind='bar',
        title='Saldo CAGED por Mes - 2025'
    )
    ax.set_ylabel('Saldo')
    
    # Formatacao do eixo Y para milhar (k)
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))
    
    plt.show()

except Exception as e:
    print(f"Erro: {e}")

## 6. SIDRA - Series IBGE

In [ ]:
# Indicadores disponiveis
print("Indicadores SIDRA:")
for ind in adb.sidra.available():
    info = adb.sidra.info(ind)
    print(f"  - {ind}: {info.get('name', 'N/A')}")

In [ ]:
# Cobertura temporal
print("SIDRA - Cobertura Temporal")
print("-" * 70)

for ind in adb.sidra.available():
    df = adb.sidra.read(ind)
    if not df.empty:
        print(f"{ind:20} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>6,} registros")
    else:
        print(f"{ind:20} | SEM DADOS")

In [ ]:
# PNAD Continua - Taxa de Desocupacao
try:
    df_pnad = adb.sidra.read('pnad_desocupacao')
    
    if not df_pnad.empty:
        print(f"PNAD Desocupacao: {len(df_pnad)} trimestres")
        
        # Visualizacao (sem passar ylabel como argumento)
        ax = df_pnad.agora.plot(
            title="Taxa de Desocupacao (PNAD Continua)"
        )
        # Definindo o label explicitamente no eixo
        ax.set_ylabel("Taxa (%)")
        
        # Adicionar destaque para o ultimo valor disponivel
        if not df_pnad.empty:
            ultimo_valor = df_pnad.iloc[-1].item()
            ultima_data = df_pnad.index[-1]
            
            ax.annotate(f'{ultimo_valor:.1f}%', 
                       xy=(ultima_data, ultimo_valor),
                       xytext=(10, 0), 
                       textcoords='offset points',
                       color='#00464D',
                       fontweight='bold',
                       va='center')
        
        plt.show()
    else:
        print("Dados da PNAD nao encontrados. Execute adb.sidra.collect() ou verifique a conexao.")

except Exception as e:
    print(f"Erro ao processar PNAD: {e}")

## 7. Bloomberg (se disponivel)

In [ ]:
# Indicadores disponiveis
print("Indicadores Bloomberg:")
for ind in adb.bloomberg.available():
    info = adb.bloomberg.info(ind)
    print(f"  - {ind}: {info.get('ticker', 'N/A')} ({info.get('category', 'N/A')})")

In [ ]:
# Verificar se tem dados coletados
try:
    df_ibov = adb.bloomberg.read('ibov_points')
    if not df_ibov.empty:
        print(f"Ibovespa: {len(df_ibov):,} registros")
        print(f"Periodo: {df_ibov.index.min().strftime('%Y-%m-%d')} a {df_ibov.index.max().strftime('%Y-%m-%d')}")
        display(df_ibov.tail())
        
        # Visualizacao corrigida
        ax = df_ibov.agora.plot(
            title="Ibovespa (Pontos)"
        )
        ax.set_ylabel("Pontos") # Setado externamente
        plt.show()
    else:
        print("Nenhum dado coletado. Requer Bloomberg Terminal.")
except Exception as e:
    print(f"Bloomberg nao disponivel: {e}")

## 8. QueryEngine - SQL Direto

Para queries mais complexas, use o QueryEngine com SQL direto via DuckDB.

In [ ]:
# Exemplo: Media anual da Selic
df_media_selic = qe.sql("""
    SELECT 
        strftime(date, '%Y') as ano,
        AVG(value) as media_selic,
        MIN(value) as min_selic,
        MAX(value) as max_selic
    FROM '{raw}/bacen/sgs/daily/selic.parquet'
    GROUP BY ano
    ORDER BY ano
""")

print("Media Anual da Selic:")
display(df_media_selic.tail(10))

In [ ]:
# Exemplo: Agregacao CAGED com SQL
try:
    df_setor = qe.sql("""
        SELECT 
            secao as setor,
            SUM(saldomovimentacao) as saldo,
            COUNT(*) as registros,
            AVG(CAST(REPLACE(salario, ',', '.') AS DOUBLE)) as salario_medio
        FROM '{raw}/mte/caged/cagedmov_2025-*.parquet'
        GROUP BY secao
        ORDER BY saldo DESC
    """)
    
    print("Saldo por Setor - CAGED 2025:")
    display(df_setor.head(10))
except Exception as e:
    print(f"Erro: {e}")